In [1]:
import pandas as pd
import numpy as np

In [2]:
BaData =pd.read_excel('British Airways.xlsx')

In [3]:
print(BaData.shape)

(10000, 17)


In [4]:
BaData.head().T

,0,1,2,3,4
FLIGHT_DATE,2025-09-02 00:00:00,2025-06-10 00:00:00,2025-10-27 00:00:00,2025-06-15 00:00:00,2025-08-25 00:00:00
FLIGHT_TIME,14:19:00,06:42:00,15:33:00,18:29:00,20:35:00
TIME_OF_DAY,Afternoon,Morning,Afternoon,Evening,Evening
AIRLINE_CD,BA,BA,BA,BA,BA
FLIGHT_NO,BA5211,BA7282,BA1896,BA5497,BA1493
DEPARTURE_STATION_CD,LHR,LHR,LHR,LHR,LHR
ARRIVAL_STATION_CD,LAX,LAX,FRA,IST,FRA
ARRIVAL_COUNTRY,USA,USA,Germany,Turkey,Germany
ARRIVAL_REGION,North America,North America,Europe,Europe,Europe
HAUL,LONG,LONG,SHORT,SHORT,SHORT


In [5]:
BaData.columns.tolist()

['FLIGHT_DATE',
 'FLIGHT_TIME',
 'TIME_OF_DAY',
 'AIRLINE_CD',
 'FLIGHT_NO',
 'DEPARTURE_STATION_CD',
 'ARRIVAL_STATION_CD',
 'ARRIVAL_COUNTRY',
 'ARRIVAL_REGION',
 'HAUL',
 'AIRCRAFT_TYPE',
 'FIRST_CLASS_SEATS',
 'BUSINESS_CLASS_SEATS',
 'ECONOMY_SEATS',
 'TIER1_ELIGIBLE_PAX',
 'TIER2_ELIGIBLE_PAX',
 'TIER3_ELIGIBLE_PAX']

In [6]:
BaData.dtypes

FLIGHT_DATE             datetime64[ns]
FLIGHT_TIME                     object
TIME_OF_DAY                     object
AIRLINE_CD                      object
FLIGHT_NO                       object
DEPARTURE_STATION_CD            object
ARRIVAL_STATION_CD              object
ARRIVAL_COUNTRY                 object
ARRIVAL_REGION                  object
HAUL                            object
AIRCRAFT_TYPE                   object
FIRST_CLASS_SEATS                int64
BUSINESS_CLASS_SEATS             int64
ECONOMY_SEATS                    int64
TIER1_ELIGIBLE_PAX               int64
TIER2_ELIGIBLE_PAX               int64
TIER3_ELIGIBLE_PAX               int64
dtype: object

In [7]:
print(BaData['HAUL'].unique())
print(BaData['ARRIVAL_REGION'].unique())

['LONG' 'SHORT']
['North America' 'Europe' 'Asia' 'Middle East']


In [8]:
print(BaData['ARRIVAL_COUNTRY'].unique())

['USA' 'Germany' 'Turkey' 'Austria' 'Netherlands' 'Japan' 'France'
 'Switzerland' 'Spain' 'UAE']


In [9]:
BaData['YEAR'] = BaData['FLIGHT_DATE'].dt.year
print(BaData['YEAR'].value_counts().sort_index())
print(f"\From {BaData['YEAR'].min()} to {BaData['YEAR'].max()}")

YEAR
2025    10000
Name: count, dtype: int64
\From 2025 to 2025


In [10]:
# Extract month
BaData['MONTH'] = BaData['FLIGHT_DATE'].dt.month

print(BaData['MONTH'].value_counts().sort_index())

MONTH
4     1436
5     1488
6     1483
7     1389
8     1429
9     1430
10    1345
Name: count, dtype: int64


In [11]:
sample= BaData[BaData['MONTH'].isin([4,7,10])].copy()
sample.shape

(4170, 19)

In [12]:
print("TIME_OF_DAY:", sample['TIME_OF_DAY'].unique())
print("HAUL:", sample['HAUL'].unique())
print("ARRIVAL_REGION:", sample['ARRIVAL_REGION'].unique())

TIME_OF_DAY: ['Afternoon' 'Evening' 'Morning' 'Lunchtime']
HAUL: ['SHORT' 'LONG']
ARRIVAL_REGION: ['Europe' 'North America' 'Middle East' 'Asia']


In [13]:
sample.head().T

,2,5,7,16,22
FLIGHT_DATE,2025-10-27 00:00:00,2025-07-12 00:00:00,2025-04-24 00:00:00,2025-04-02 00:00:00,2025-04-10 00:00:00
FLIGHT_TIME,15:33:00,19:08:00,14:50:00,17:04:00,16:04:00
TIME_OF_DAY,Afternoon,Evening,Afternoon,Afternoon,Afternoon
AIRLINE_CD,BA,BA,BA,BA,BA
FLIGHT_NO,BA1896,BA4954,BA7116,BA8677,BA1609
DEPARTURE_STATION_CD,LHR,LHR,LHR,LHR,LHR
ARRIVAL_STATION_CD,FRA,VIE,ORD,ORD,DXB
ARRIVAL_COUNTRY,Germany,Austria,USA,USA,UAE
ARRIVAL_REGION,Europe,Europe,North America,North America,Middle East
HAUL,SHORT,SHORT,LONG,LONG,LONG


In [14]:
sample['TOTAL_SEATS'] = (sample['FIRST_CLASS_SEATS'] + sample['BUSINESS_CLASS_SEATS'] + sample['ECONOMY_SEATS'])

In [16]:
sample['TIER1_%'] = (sample['TIER1_ELIGIBLE_PAX']/ sample['TOTAL_SEATS']* 100).round(1)
sample['TIER2_%'] = (sample['TIER2_ELIGIBLE_PAX']/ sample['TOTAL_SEATS']* 100).round(1)
sample['TIER3_%'] = (sample['TIER3_ELIGIBLE_PAX']/ sample['TOTAL_SEATS']* 100).round(1)

In [17]:
lookup = sample.groupby(['TIME_OF_DAY', 'HAUL', 'ARRIVAL_REGION']).agg(
    Tier1_pct = ('TIER1_%', 'mean'),
    Tier2_pct = ('TIER2_%', 'mean'),
    Tier3_pct = ('TIER3_%', 'mean'),
    Flight_Count = ('FLIGHT_NO', 'count')
    ).round(1).reset_index()

print(lookup)

   TIME_OF_DAY   HAUL ARRIVAL_REGION  Tier1_pct  Tier2_pct  Tier3_pct  \
0    Afternoon   LONG           Asia        0.2        2.7       10.3   
1    Afternoon   LONG    Middle East        0.3        3.0       11.1   
2    Afternoon   LONG  North America        0.2        3.0       11.4   
3    Afternoon  SHORT         Europe        0.4        4.4       16.9   
4      Evening   LONG           Asia        0.2        3.1       11.7   
5      Evening   LONG    Middle East        0.2        3.0       11.3   
6      Evening   LONG  North America        0.2        2.9       11.1   
7      Evening  SHORT         Europe        0.3        4.3       16.5   
8    Lunchtime   LONG           Asia        0.1        2.6       10.1   
9    Lunchtime   LONG    Middle East        0.2        3.3       12.3   
10   Lunchtime   LONG  North America        0.2        2.8       10.9   
11   Lunchtime  SHORT         Europe        0.4        4.5       17.1   
12     Morning   LONG           Asia        0.2    

In [18]:
# Add a justification sheet too
justification = pd.DataFrame({
    'Decision': [
        'Sample Selection',
        'Time of Day',
        'Haul Type',
        'Region',
        'Tier Percentages'
    ],
    'Justification': [
        'Selected April, July, October to represent spring, peak summer and autumn travel patterns',
        'Four categories: Morning, Lunchtime, Afternoon, Evening based on actual flight data',
        'SHORT vs LONG haul — short haul dominated by Europe economy travellers, long haul has more premium seats',
        'Europe, North America, Middle East, Asia — each region has distinct passenger profile',
        'Percentages derived from actual TIER_ELIGIBLE_PAX divided by total seats — data driven not assumed'
    ]
})

# Export both to Excel with two sheets
with pd.ExcelWriter('BA_Lounge_Eligibility.xlsx') as writer:
    lookup.to_excel(writer, sheet_name='Lookup_Table', index=False)
    justification.to_excel(writer, sheet_name='Justification', index=False)

print('File saved!')

File saved!
